# Local TOFU: learning, forgetting, and recovery after quantization

1. Load the local Llama model.
2. Fine-tune on the complete TOFU full split and save the complete model separately.
3. Reload and evaluate the fine-tuned checkpoint: establish what it can answer.
4. Apply NPO + retain regularization to a copy loaded from the saved SFT model.
5. Evaluate the unlearned FP16 model on the same questions.
6. Quantize that unlearned checkpoint to NF4, save/reload it, and ask the same questions again.
7. Read the paired report: exact-answer losses/returns, score changes, retained utility, and all generated answers.

Kernel: `C:/VENV/fyp/Scripts/python.exe`. Model: `C:/Model/Llama-3.2-1B-Instruct`.
This notebook includes its evaluation/report code and remains standalone.


## 0 · Verify the existing fyp kernel

In [1]:
# Use the existing environment: C:/VENV/fyp/Scripts/python.exe
import sys
print('Notebook interpreter:', sys.executable)


Notebook interpreter: c:\VENV\fyp\Scripts\python.exe


## 1 · Paths and experiment settings
**Reopen an experiment:** set `RUN_MODE = 'continue'` and `REUSE_SFT_MODEL` to that
run's `sft_full_fp16` folder. All outputs use its parent run folder. Completed NPO,
merged FP16 and NF4 models are reused; missing stages are created there. Evaluation
cells still run when executed. Saved settings and dataset snapshots are preserved.

**Different unlearning settings / repeat NPO:** use `RUN_MODE = 'new'` with the same
SFT path. **Fresh SFT:** set both model/resume paths to `None`; this creates a new run.
SFT power-cut recovery continues to use a new run to preserve its checkpoint source.


In [ ]:
import gc, hashlib, json, math, random, re
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import get_cosine_schedule_with_warmup


# Select C:/VENV/fyp/Scripts/python.exe as your notebook kernel.
MODEL_PATH = Path(r'C:\Model\Llama-3.2-1B-Instruct')
# 'continue' keeps the saved SFT run folder; 'new' starts a separate experiment.
RUN_MODE = 'continue'
# Select the complete SFT model to reuse below; leave None for fresh training.
# e.g. Path(r'C:\MY-FYP\fq-ssiuu\runs\YOUR_RUN\sft_full_fp16')
REUSE_SFT_MODEL = Path(r'C:\MY-FYP\fq-ssiuu\runs\local_tofu_npo_20261025_120840_678375\sft_full_fp16')
# None starts fresh in a NEW run. To recover, paste the interrupted run folder.
SFT_RESUME_FROM = None  # e.g. Path(r'C:\MY-FYP\fq-ssiuu\runs\YOUR_INTERRUPTED_RUN')
SFT_SAVE_EVERY = 50     # optimizer updates, not individual examples
SFT_KEEP_CHECKPOINTS = 2  # newest two complete recovery checkpoints per run
SEED = 42
MAX_LEN = 256
SFT_EPOCHS = 8          # every row of TOFU full is visited once per epoch
SFT_MAX_STEPS = None   # leave None for complete training; smoke tests override this
NPO_STEPS = 60
GRAD_ACCUM = 8
LR_SFT = 2e-4
LR_NPO = 5e-5
BETA = 0.1
RETAIN_COEFF = 1.0
FORGET_SPLIT = 'forget05'  # choices: forget01, forget05, forget10
assert FORGET_SPLIT in ('forget01','forget05','forget10')
RETAIN_SPLIT = f'retain{100-int(FORGET_SPLIT[-2:]):02d}'
EVAL_FORGET_LIMIT = None   # None evaluates every forget question
EVAL_PER_SPLIT = 40       # fixed samples for retain/world utility
EVAL_NEW_TOKENS = 128
SFT_MICRO_BATCH_SIZE = 2

# The optional command-line runner supplies overrides; normal notebook execution needs none.
options = globals().get('LOCAL_RUN_OPTIONS', {})
RUN_MODE = options.get('run_mode') or RUN_MODE
assert RUN_MODE in ('continue', 'new')
if options.get('smoke'):
    SFT_MAX_STEPS, NPO_STEPS, GRAD_ACCUM = 1, 1, 1
    EVAL_PER_SPLIT, EVAL_NEW_TOKENS = 2, 16
    EVAL_FORGET_LIMIT = 2
if options.get('reuse_sft_model'):
    SFT_RESUME_FROM = None
    REUSE_SFT_MODEL = Path(options['reuse_sft_model'])
if options.get('resume_sft'):
    REUSE_SFT_MODEL = None
    SFT_RESUME_FROM = Path(options['resume_sft'])
if options.get('fresh_sft'):
    SFT_RESUME_FROM = None
    REUSE_SFT_MODEL = None
assert not (SFT_RESUME_FROM is not None and REUSE_SFT_MODEL is not None), 'Choose resume training OR reuse a finished model'
assert SFT_SAVE_EVERY > 0 and SFT_KEEP_CHECKPOINTS >= 2
if REUSE_SFT_MODEL is not None:
    REUSE_SFT_MODEL = Path(REUSE_SFT_MODEL).resolve()
    assert (REUSE_SFT_MODEL/'config.json').is_file(), 'Saved full SFT model not found'
else:
    assert MODEL_PATH.is_dir(), f'Local model path not found: {MODEL_PATH}'
assert SFT_EPOCHS > 0 and GRAD_ACCUM > 0
assert SFT_MAX_STEPS is None or SFT_MAX_STEPS > 0
assert torch.cuda.is_available(), 'CUDA GPU required'
print('Torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU:', torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print(f'Free GPU memory: {free/2**30:.2f} / {total/2**30:.2f} GiB')
assert free > 6 * 2**30, 'Release models in other notebook kernels before training (need at least 6 GiB free).'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
ROOT = Path.cwd()

def select_experiment_dir(root, reuse_model, run_mode, options):
    if options.get('resume_eval'):
        folder = Path(options['resume_eval']).resolve()
        assert (folder/'merged_fp16'/'config.json').is_file()
        return folder, True
    if run_mode == 'continue' and reuse_model is not None:
        folder = Path(reuse_model).resolve().parent
        if Path(reuse_model).name != 'sft_full_fp16' or not (folder/'settings.json').is_file():
            raise ValueError("Continue requires this notebook's run/sft_full_fp16 folder; use RUN_MODE='new' for an external model.")
        return folder, True
    folder = Path(root)/'runs'/('local_tofu_npo_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
    folder.mkdir(parents=True, exist_ok=False)
    return folder, False

EXPERIMENT_DIR, CONTINUING_RUN = select_experiment_dir(ROOT, REUSE_SFT_MODEL, RUN_MODE, options)

def saved_stage_exists(folder, adapter=False):
    """Missing stage may be created; an incomplete existing stage is never overwritten."""
    folder = Path(folder)
    if not folder.exists():
        return False
    config = folder/('adapter_config.json' if adapter else 'config.json')
    index = folder/'model.safetensors.index.json'
    if not config.is_file():
        raise RuntimeError(f'Incomplete stage: {folder}. Preserve it and use a new run to rebuild.')
    json.loads(config.read_text(encoding='utf-8'))
    if adapter:
        names = ['adapter_model.safetensors']
    elif index.is_file():
        names = list(set(json.loads(index.read_text(encoding='utf-8'))['weight_map'].values()))
    else:
        names = ['model.safetensors']
    if not names or any(not (folder/name).is_file() or (folder/name).stat().st_size == 0 for name in names):
        raise RuntimeError(f'Missing or empty weights in {folder}. Use a new run to rebuild.')
    if not (folder/'tokenizer.json').is_file():
        raise RuntimeError(f'Missing tokenizer in {folder}. Use a new run to rebuild.')
    return True

def verify_continuation_settings(saved, current):
    keys = ('model_path', 'sft_model_dir', 'seed', 'max_length', 'npo_steps', 'accumulation',
            'lr_npo', 'beta', 'retain_coeff', 'eval_per_split', 'eval_new_tokens',
            'forget_split', 'retain_split', 'eval_forget_limit', 'smoke_only')
    changed = []
    for key in keys:
        a, b = saved.get(key), current.get(key)
        if key in ('model_path', 'sft_model_dir') and a and b:
            a, b = Path(a).resolve(), Path(b).resolve()
        if a != b:
            changed.append(key)
    if changed:
        raise ValueError(f"Settings differ from this saved experiment: {changed}. Restore them or set RUN_MODE='new'.")

SFT_ADAPTER = EXPERIMENT_DIR/'sft_adapter'
SFT_MODEL_DIR = REUSE_SFT_MODEL or EXPERIMENT_DIR/'sft_full_fp16'
NPO_ADAPTER = EXPERIMENT_DIR/'npo_adapter'
MERGED_DIR = EXPERIMENT_DIR/'merged_fp16'
settings = dict(model_path=str(MODEL_PATH), sft_model_dir=str(SFT_MODEL_DIR),
    method='FP16 LoRA SFT on full TOFU, merged SFT checkpoint, fresh NPO+retain LoRA, NF4',
    seed=SEED, max_length=MAX_LEN, sft_epochs=SFT_EPOCHS, sft_max_steps=SFT_MAX_STEPS,
    sft_resume_from=str(SFT_RESUME_FROM) if SFT_RESUME_FROM else None,
    sft_save_every=SFT_SAVE_EVERY, sft_keep_checkpoints=SFT_KEEP_CHECKPOINTS,
    npo_steps=NPO_STEPS, accumulation=GRAD_ACCUM, lr_sft=LR_SFT, lr_npo=LR_NPO,
    beta=BETA, retain_coeff=RETAIN_COEFF, eval_per_split=EVAL_PER_SPLIT,
    eval_new_tokens=EVAL_NEW_TOKENS, forget_split=FORGET_SPLIT, retain_split=RETAIN_SPLIT,
    eval_forget_limit=EVAL_FORGET_LIMIT, smoke_only=bool(options.get('smoke')))
if CONTINUING_RUN and not options.get('resume_eval'):
    saved_settings = json.loads((EXPERIMENT_DIR/'settings.json').read_text(encoding='utf-8'))
    verify_continuation_settings(saved_settings, settings)
if not CONTINUING_RUN:
    (EXPERIMENT_DIR/'settings.json').write_text(json.dumps(settings, indent=2), encoding='utf-8')
import importlib.metadata
if not CONTINUING_RUN:
    (EXPERIMENT_DIR/'environment.json').write_text(json.dumps({p: importlib.metadata.version(p) for p in ['torch','transformers','peft','datasets','bitsandbytes']}, indent=2), encoding='utf-8')
print('Experiment:', EXPERIMENT_DIR)
print('Complete fine-tuned model:', SFT_MODEL_DIR)

print('Run mode:', 'continue existing experiment' if CONTINUING_RUN else 'new experiment')


## 2 · Download TOFU splits; validate columns and overlap

In [ ]:
from datasets import load_from_disk

DATA_DIR = Path(r'C:\MY-FYP\fq-ssiuu\data\tofu')
DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = DATA_DIR/'hf_cache'

def get_split(name):
    # Evaluation-only resume uses this run's exact saved dataset snapshots.
    snapshot=EXPERIMENT_DIR/f'dataset_{name}.jsonl'
    if CONTINUING_RUN:
        assert snapshot.is_file(), f'Missing saved dataset: {snapshot}'
        return load_dataset('json',data_files=str(snapshot),split='train')
    local=DATA_DIR/name
    if local.exists():
        return load_from_disk(str(local))
    ds=load_dataset('locuslab/TOFU',name,split='train',cache_dir=str(CACHE_DIR))
    ds.save_to_disk(str(local))
    return ds

full,forget,retain,world=[get_split(name) for name in ('full',FORGET_SPLIT,RETAIN_SPLIT,'world_facts')]
for name,ds in [('full',full),(FORGET_SPLIT,forget),(RETAIN_SPLIT,retain),('world_facts',world)]:
    assert {'question','answer'}.issubset(ds.column_names)
    print(name,len(ds))
key=lambda row:(row['question'].strip(),row['answer'].strip())
assert {key(x) for x in forget}.isdisjoint({key(x) for x in retain}), 'forget and retain sets overlap'
assert {key(x) for x in full}=={key(x) for x in forget}|{key(x) for x in retain}
if not CONTINUING_RUN:
    for name,ds in [('full',full),(FORGET_SPLIT,forget),(RETAIN_SPLIT,retain),('world_facts',world)]:
        ds.to_json(str(EXPERIMENT_DIR/f'dataset_{name}.jsonl'))
full_dataset_sha256=hashlib.sha256(json.dumps(list(full),sort_keys=True,ensure_ascii=False).encode('utf-8')).hexdigest()
print('Verified disjoint partition:',FORGET_SPLIT,'+',RETAIN_SPLIT,'= full TOFU')


## 3 · Tokenizer, answer-only supervision, and FP16 LoRA model

In [4]:
tokenizer = AutoTokenizer.from_pretrained(str(SFT_MODEL_DIR if REUSE_SFT_MODEL else MODEL_PATH), local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def prompt_text(q):
    messages=[{'role':'user','content':q}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def encode_qa(example):
    prompt = prompt_text(example['question'])
    completion = example['answer'] + (tokenizer.eos_token or '')
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    answer_ids = tokenizer(completion, add_special_tokens=False)['input_ids']
    # Preserve answer tokens when truncating; avoid empty supervised response.
    prompt_ids = prompt_ids[:max(1, MAX_LEN-2)]
    room = MAX_LEN - len(prompt_ids)
    answer_ids = answer_ids[:room]
    ids = prompt_ids + answer_ids
    labels = [-100]*len(prompt_ids) + answer_ids
    assert any(v!=-100 for v in labels)
    return {'input_ids':ids,'attention_mask':[1]*len(ids),'labels':labels}

def collate(rows):
    longest=max(len(r['input_ids']) for r in rows)
    out={}
    for key,pad in [('input_ids',tokenizer.pad_token_id),('attention_mask',0),('labels',-100)]:
        out[key]=torch.tensor([r[key]+[pad]*(longest-len(r[key])) for r in rows],dtype=torch.long,device='cuda')
    return out

qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                           bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
def load_fp16(path=MODEL_PATH):
    return AutoModelForCausalLM.from_pretrained(str(path), 
                device_map='cpu', torch_dtype=torch.float16, local_files_only=True,
                attn_implementation='sdpa').to('cuda')

lora_config=LoraConfig(r=32,lora_alpha=32,lora_dropout=0.0,bias='none',task_type='CAUSAL_LM',
    # target_modules=['q_proj','k_proj','v_proj','o_proj']
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
    )

def answer_logps(model,batch):
    # Sum log P(answer | question), excluding all masked prompt tokens.
    logits=model(input_ids=batch['input_ids'],attention_mask=batch['attention_mask']).logits
    targets=batch['labels'][:,1:]
    valid=targets.ne(-100)
    losses=F.cross_entropy(logits[:, :-1, :].float().reshape(-1, logits.shape[-1]),
                           targets.reshape(-1), ignore_index=-100, reduction='none')
    return -losses.reshape_as(targets).sum(dim=-1), valid.sum(dim=-1).clamp(min=1)

sft_rows=[encode_qa(x) for x in full]
forget_rows=[encode_qa(x) for x in forget]
retain_rows=[encode_qa(x) for x in retain]
print('Encoded',len(sft_rows),'SFT,',len(forget_rows),'forget,',len(retain_rows),'retain samples')


Encoded 4000 SFT, 200 forget, 3800 retain samples


## 4. Fine-tune full TOFU, with power-cut recovery

Every 50 optimizer steps, save the adapter, optimizer, cosine scheduler, random states,
and exact data position under this run's `sft_checkpoints/`. Keep the newest two complete saves.
The final step is also saved. An interrupted write is ignored; corrupt latest saves fall back
to the previous intact save when a run folder is selected.

**After a restart:** set `SFT_RESUME_FROM = Path(r'...\runs\YOUR_INTERRUPTED_RUN')`
and `REUSE_SFT_MODEL = None` in configuration, then execute configuration, data, helpers,
and this cell. Resume uses a new output run and preserves the source run. Keep the original
training settings. For a second interruption, select the newest run containing a checkpoint.

**Train from scratch:** set both paths to `None` and rerun from configuration to get a new run.
**Skip training:** set only `REUSE_SFT_MODEL` to a finished `sft_full_fp16` folder.
Checkpoints from older notebook versions cannot resume training.

An already running cell does not pick up notebook edits. No saved checkpoint exists before
the first save; ordinary interruptions lose up to 49 completed updates plus in-progress work.


In [5]:
# Embedded verbatim in Mother_Local_v2.ipynb; the notebook needs no script imports.
import os
import shutil
import uuid


def checkpoint_digest(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def find_sft_checkpoint(location):
    """Accept a run, its sft_checkpoints folder, or one explicit checkpoint."""
    location = Path(location).resolve()
    explicit = location.name.startswith('step-')
    folder = location/'sft_checkpoints' if (location/'sft_checkpoints').is_dir() else location
    candidates = [location] if explicit else sorted(folder.glob('step-*'), reverse=True)
    for candidate in candidates:
        try:
            manifest = json.loads((candidate/'complete.json').read_text(encoding='utf-8'))
            required = {'adapter/adapter_config.json', 'adapter/adapter_model.safetensors', 'training_state.pt'}
            if not required.issubset(manifest['sha256']):
                raise ValueError('Missing required checkpoint files')
            for name, digest in manifest['sha256'].items():
                path = (candidate/name).resolve()
                if not path.is_relative_to(candidate.resolve()) or checkpoint_digest(path) != digest:
                    raise ValueError(f'Checkpoint integrity check failed: {name}')
            return candidate
        except (OSError, ValueError, KeyError, TypeError) as error:
            print('Skipping invalid checkpoint:', candidate, '|', error)
    raise FileNotFoundError(f'No complete, intact SFT checkpoint found in {location}')


def save_sft_checkpoint(state, policy, tokenizer):
    """Publish a new directory only after every file has been flushed and hashed."""
    root = (EXPERIMENT_DIR/'sft_checkpoints').resolve()
    root.mkdir(parents=True, exist_ok=True)
    destination = root/f"step-{state['step']:08d}"
    if destination.exists():
        raise FileExistsError(f'Checkpoint already exists: {destination}; use a new run')
    temporary = root/f'.incomplete-{uuid.uuid4().hex}'
    temporary.mkdir()
    policy.save_pretrained(str(temporary/'adapter'), safe_serialization=True)
    tokenizer.save_pretrained(str(temporary/'adapter'))
    torch.save(state, temporary/'training_state.pt')
    digests = {}
    for path in sorted(temporary.rglob('*')):
        if path.is_file():
            with path.open('rb+') as stream:
                os.fsync(stream.fileno())
            digests[path.relative_to(temporary).as_posix()] = checkpoint_digest(path)
    with (temporary/'complete.json').open('w', encoding='utf-8') as stream:
        json.dump({'step': state['step'], 'sha256': digests}, stream, indent=2)
        stream.flush()
        os.fsync(stream.fileno())
    temporary.rename(destination)
    print('Saved recovery checkpoint:', destination)
    # Retain two committed saves by default. Never prune the source resume run.
    previous = sorted(root.glob('step-*'), reverse=True)
    for old in previous[SFT_KEEP_CHECKPOINTS:]:
        resolved = old.resolve()
        if resolved.parent != root or old.is_symlink() or not (old/'complete.json').is_file():
            raise RuntimeError(f'Refusing to prune unexpected checkpoint path: {old}')
        shutil.rmtree(resolved)
    return destination


if REUSE_SFT_MODEL is not None:
    training_record = json.loads((SFT_MODEL_DIR/'sft_training.json').read_text(encoding='utf-8'))
    assert training_record['dataset_sha256'] == full_dataset_sha256, 'Saved SFT used different TOFU rows'
    assert training_record['completed_full_epochs'] >= 1, 'This is an incomplete/smoke checkpoint; complete SFT first'
    assert training_record['max_length'] == MAX_LEN, 'Match MAX_LEN to the saved SFT training settings'
    print('Reusing complete SFT checkpoint:', SFT_MODEL_DIR)
else:
    if SFT_MODEL_DIR.exists() or SFT_ADAPTER.exists():
        raise FileExistsError('SFT output already exists. Rerun configuration for a new run; optionally set SFT_RESUME_FROM.')
    assert SFT_SAVE_EVERY > 0 and SFT_KEEP_CHECKPOINTS >= 2
    total_steps = SFT_EPOCHS * math.ceil(len(sft_rows)/GRAD_ACCUM)
    target_steps = min(total_steps, SFT_MAX_STEPS) if SFT_MAX_STEPS is not None else total_steps
    # Encoded-data identity also detects tokenizer, template and truncation changes.
    signature = dict(version=1, dataset_sha256=full_dataset_sha256,
        encoded_sha256=hashlib.sha256(json.dumps(sft_rows, sort_keys=True).encode()).hexdigest(),
        model_path=str(MODEL_PATH.resolve()), max_length=MAX_LEN, seed=SEED,
        epochs=SFT_EPOCHS, max_steps=SFT_MAX_STEPS, accumulation=GRAD_ACCUM, lr=LR_SFT,
        lora=json.loads(json.dumps(lora_config.to_dict(), default=lambda x: sorted(x) if isinstance(x, set) else str(x))),
        schedule='cosine_5pct_warmup', total_steps=total_steps)
    resume_state = None
    resumed_from = None
    if SFT_RESUME_FROM is not None:
        resumed_from = find_sft_checkpoint(SFT_RESUME_FROM)
        # This stores only tensors and basic Python data; never unpickle arbitrary objects.
        resume_state = torch.load(resumed_from/'training_state.pt', map_location='cpu', weights_only=True)
        if resume_state['signature'] != signature:
            changed = [key for key in signature if resume_state['signature'].get(key) != signature[key]]
            raise ValueError(f'Resume settings/data differ: {changed}. Restore the original settings or start fresh.')
        policy = PeftModel.from_pretrained(load_fp16(MODEL_PATH), str(resumed_from/'adapter'), is_trainable=True)
    else:
        policy = get_peft_model(load_fp16(MODEL_PATH), lora_config)
    policy.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    policy.enable_input_require_grads()
    policy.config.use_cache = False
    policy.print_trainable_parameters()
    optimizer = torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad], lr=LR_SFT)
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.05*total_steps), total_steps)
    rng = random.Random(SEED)
    step = 0; examples_seen = 0; completed_full_epochs = 0; sft_history = []
    epoch = 0; position = 0; order = None
    if resume_state is not None:
        optimizer.load_state_dict(resume_state['optimizer'])
        scheduler.load_state_dict(resume_state['scheduler'])
        step = resume_state['step']; examples_seen = resume_state['examples_seen']
        completed_full_epochs = resume_state['completed_full_epochs']
        epoch = resume_state['epoch']; position = resume_state['position']; order = resume_state['order']
        sft_history = resume_state['history']
        rng.setstate(resume_state['shuffle_rng'])
        random.setstate(resume_state['python_rng'])
        nr = resume_state['numpy_rng']
        np.random.set_state((nr[0], np.array(nr[1], dtype=np.uint32), nr[2], nr[3], nr[4]))
        torch.set_rng_state(resume_state['torch_rng'])
        if torch.cuda.is_available():
            torch.cuda.set_rng_state_all(resume_state['cuda_rng'])
        if step < target_steps:
            print(f'Resuming after optimizer step {step}; next step {step+1}, epoch {epoch+1}')
        else:
            print(f'All {step} optimizer steps are saved; finishing model export.')
        # Give the new run its own recovery point even before its next 50-step save.
        save_sft_checkpoint(resume_state, policy, tokenizer)
        del resume_state
    policy.train()
    while epoch < SFT_EPOCHS and step < target_steps:
        if order is None:
            order = list(range(len(sft_rows))); rng.shuffle(order)
        indices = order[position:position+GRAD_ACCUM]
        optimizer.zero_grad(set_to_none=True)
        mean_loss = 0.0
        for offset in range(0, len(indices), SFT_MICRO_BATCH_SIZE):
            batch_indices = indices[offset:offset + SFT_MICRO_BATCH_SIZE]
            batch = collate([sft_rows[index] for index in batch_indices])
            logps, token_count = answer_logps(policy, batch)
            loss = -(logps / token_count).sum() / len(indices)

            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite SFT loss')
            loss.backward()
            mean_loss += float(loss.detach())
        torch.nn.utils.clip_grad_norm_([p for p in policy.parameters() if p.requires_grad], 1.0, error_if_nonfinite=True)
        optimizer.step(); scheduler.step()
        step += 1; position += len(indices); examples_seen += len(indices)
        if step == 1 or step % 10 == 0:
            sft_history.append({'step': step, 'epoch': epoch+1, 'loss_per_token': mean_loss})
            print(f'SFT {step}/{total_steps}, epoch {epoch+1}, loss/token {mean_loss:.4f}')
        if position == len(sft_rows):
            completed_full_epochs += 1; epoch += 1; position = 0; order = None
        # Save only at a completed optimizer update, including the final partial batch.
        if step % SFT_SAVE_EVERY == 0 or step == target_steps:
            nr = np.random.get_state()
            state = dict(signature=signature, optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                step=step, epoch=epoch, position=position, order=order, examples_seen=examples_seen,
                completed_full_epochs=completed_full_epochs, history=sft_history,
                shuffle_rng=rng.getstate(), python_rng=random.getstate(),
                numpy_rng=(nr[0], nr[1].tolist(), nr[2], nr[3], nr[4]),
                torch_rng=torch.get_rng_state(),
                cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])
            save_sft_checkpoint(state, policy, tokenizer)
            del state
    if SFT_MAX_STEPS is None:
        assert completed_full_epochs == SFT_EPOCHS
        assert examples_seen == SFT_EPOCHS*len(sft_rows)
    policy.save_pretrained(str(SFT_ADAPTER))
    tokenizer.save_pretrained(str(SFT_ADAPTER))
    (EXPERIMENT_DIR/'sft_history.json').write_text(json.dumps(sft_history, indent=2))
    # The scheduler holds the optimizer; release both before loading the merge base.
    del scheduler, optimizer, policy
    for name in ('loss', 'logps', 'token_count', 'batch'):
        globals().pop(name, None)
    gc.collect(); torch.cuda.empty_cache()

    base_cpu = AutoModelForCausalLM.from_pretrained(str(MODEL_PATH), dtype=torch.float16,
        device_map='cpu', local_files_only=True)
    sft_merged = PeftModel.from_pretrained(base_cpu, str(SFT_ADAPTER), is_trainable=False).merge_and_unload(safe_merge=True)
    sft_merged.config.use_cache = True
    sft_merged.save_pretrained(str(SFT_MODEL_DIR), safe_serialization=True, max_shard_size='2GB')
    tokenizer.save_pretrained(str(SFT_MODEL_DIR))
    training_record = dict(dataset='locuslab/TOFU', split='full', dataset_rows=len(sft_rows),
        dataset_sha256=full_dataset_sha256, examples_seen=examples_seen,
        completed_full_epochs=completed_full_epochs, optimizer_steps=step,
        max_length=MAX_LEN, seed=SEED, base_model=str(MODEL_PATH),
        checkpoint_type='merged_full_model_fp16', smoke_only=bool(options.get('smoke')),
        resumed_from=str(resumed_from) if resumed_from else None)
    (SFT_MODEL_DIR/'sft_training.json').write_text(json.dumps(training_record, indent=2), encoding='utf-8')
    del base_cpu, sft_merged; gc.collect()
    print('Saved complete fine-tuned model:', SFT_MODEL_DIR)
    print('Full epochs completed:', completed_full_epochs, '| Examples seen:', examples_seen)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:08<00:00, 16.25it/s]


trainable params: 22,544,384 || all params: 1,258,358,784 || trainable%: 1.7916
SFT 1/4000, epoch 1, loss/token 2.5295
SFT 10/4000, epoch 1, loss/token 2.3085
SFT 20/4000, epoch 1, loss/token 2.0237
SFT 30/4000, epoch 1, loss/token 1.8882
SFT 40/4000, epoch 1, loss/token 1.4855
SFT 50/4000, epoch 1, loss/token 1.9201
Saved recovery checkpoint: C:\MY-FYP\fq-ssiuu\runs\local_tofu_npo_20261025_120840_678375\sft_checkpoints\step-00000050
SFT 60/4000, epoch 1, loss/token 1.6548
SFT 70/4000, epoch 1, loss/token 1.7594
SFT 80/4000, epoch 1, loss/token 1.4869
SFT 90/4000, epoch 1, loss/token 1.6068
SFT 100/4000, epoch 1, loss/token 1.9189
Saved recovery checkpoint: C:\MY-FYP\fq-ssiuu\runs\local_tofu_npo_20261025_120840_678375\sft_checkpoints\step-00000100
SFT 110/4000, epoch 1, loss/token 1.5573
SFT 120/4000, epoch 1, loss/token 2.0479
SFT 130/4000, epoch 1, loss/token 2.0225
SFT 140/4000, epoch 1, loss/token 1.8721
SFT 150/4000, epoch 1, loss/token 1.5718
Saved recovery checkpoint: C:\MY-FYP\

Writing model shards: 100%|██████████| 2/2 [00:05<00:00,  2.55s/it]


Saved complete fine-tuned model: c:\MY-FYP\fq-ssiuu\runs\local_tofu_npo_20261025_120840_678375\sft_full_fp16
Full epochs completed: 8 | Examples seen: 32000


## 5 · Evaluate the saved fine-tuned model: baseline answers before unlearning

In [ ]:
def normalize(s):
    return ' '.join(re.findall(r'\w+',s.lower()))

def f1_score(hyp, ref):
    a=normalize(hyp).split(); b=normalize(ref).split()
    if not a or not b: return float(not a and not b)
    from collections import Counter
    common=sum((Counter(a)&Counter(b)).values())
    if common==0:return 0.0
    p=common/len(a);r=common/len(b)
    return 2*p*r/(p+r)

def select_eval_sets(forget,retain,world,forget_split,retain_split,seed,utility_limit,forget_limit=None):
    def choose(ds,limit):
        if limit is None: return [dict(row) for row in ds]
        assert limit>0
        return [dict(ds[i]) for i in random.Random(seed).sample(range(len(ds)),min(limit,len(ds)))]
    return {forget_split:choose(forget,forget_limit),retain_split:choose(retain,utility_limit),
            'world_facts':choose(world,utility_limit)}

# Construct once; every model is asked exactly these questions with identical decoding.
eval_sets=select_eval_sets(forget,retain,world,FORGET_SPLIT,RETAIN_SPLIT,SEED,EVAL_PER_SPLIT,EVAL_FORGET_LIMIT)
if CONTINUING_RUN and (EXPERIMENT_DIR/'eval_samples.json').is_file():
    eval_sets=json.loads((EXPERIMENT_DIR/'eval_samples.json').read_text(encoding='utf-8'))
else:
    (EXPERIMENT_DIR/'eval_samples.json').write_text(json.dumps(eval_sets,indent=2),encoding='utf-8')
print('Evaluation counts:',{name:len(rows) for name,rows in eval_sets.items()})

def evaluate(model,label):
    model.eval(); records=[]
    print("Evaluating",label,"on the fixed question set")
    for subset, examples in eval_sets.items():
        for row_index,ex in enumerate(examples):
            enc=encode_qa(ex);b=collate([enc])
            with torch.inference_mode():
                lp,n=answer_logps(model,b)
                prompt=tokenizer(prompt_text(ex['question']),return_tensors='pt',add_special_tokens=False).to('cuda')
                ids=model.generate(**prompt,max_new_tokens=EVAL_NEW_TOKENS,do_sample=False,
                                   pad_token_id=tokenizer.pad_token_id,use_cache=True)
            prediction=tokenizer.decode(ids[0,prompt['input_ids'].shape[-1]:],skip_special_tokens=True).strip()
            reference=ex['answer']
            if (row_index+1)%10==0 or row_index+1==len(examples): print(label,subset,f'{row_index+1}/{len(examples)}')
            records.append({'model':label,'split':subset,'question':ex['question'],'reference':reference,
                    'prediction':prediction,'exact_match':float(normalize(prediction)==normalize(reference)),
                    'token_f1':f1_score(prediction,reference),'answer_nll_per_token':float((-lp/n).item())})
    import pandas as pd
    df=pd.DataFrame(records)
    df.to_csv(EXPERIMENT_DIR/f'eval_{label}.csv',index=False)
    table=df.groupby('split')[['exact_match','token_f1','answer_nll_per_token']].mean()
    print('\n',label,'\n',table.round(4))
    return df

sft_model=load_fp16(SFT_MODEL_DIR)
eval_sft=evaluate(sft_model,'sft')
del sft_model;gc.collect();torch.cuda.empty_cache()




**Baseline interpretation:** Exact match is strict; token F1 is word overlap, not factual accuracy.
Inspect whether the fine-tuned model answers the target facts correctly. A question it
already gets wrong cannot establish successful forgetting. The final report conditions
exact-answer recovery on answers that were exact at SFT and lost after NPO.


## 6 · Apply NPO + retain regularization to the saved fine-tuned model

In [ ]:
# Reopen completed stages without retraining or invalidating their quantized model.
if saved_stage_exists(NPO_ADAPTER, adapter=True):
    adapter_info = json.loads((NPO_ADAPTER/'adapter_config.json').read_text(encoding='utf-8'))
    assert Path(adapter_info['base_model_name_or_path']).resolve() == SFT_MODEL_DIR.resolve(), 'NPO adapter belongs to a different SFT model'
    npo_history = json.loads((EXPERIMENT_DIR/'npo_history.json').read_text(encoding='utf-8'))
    assert npo_history and npo_history[-1]['step'] == NPO_STEPS, 'Incomplete NPO stage; use a new run to rebuild'
    print('Reusing completed NPO adapter:', NPO_ADAPTER)
else:
    if MERGED_DIR.exists() or (EXPERIMENT_DIR/'quantized_nf4').exists():
        raise RuntimeError("Downstream models exist without their NPO adapter. Use RUN_MODE='new' to rebuild together.")
    # Reference = trained SFT checkpoint, not original pretrained base.
    reference=load_fp16(SFT_MODEL_DIR)
    reference.eval()
    for p in reference.parameters(): p.requires_grad_(False)

    # Cache the frozen SFT reference scores, then free that model before NPO.
    with torch.inference_mode():
        reference_logps = [float(answer_logps(reference, collate([row]))[0].item()) for row in forget_rows]
    (EXPERIMENT_DIR/'reference_logps.json').write_text(json.dumps(reference_logps))
    del reference; gc.collect(); torch.cuda.empty_cache()

    # Reload the same complete SFT model, then attach a fresh trainable NPO adapter.
    policy=get_peft_model(load_fp16(SFT_MODEL_DIR),lora_config)
    policy.config.use_cache=False
    policy.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    policy.enable_input_require_grads()
    opt=torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad],lr=LR_NPO)
    rng_forget=random.Random(SEED+1);rng_retain=random.Random(SEED+2)
    policy.train(); opt.zero_grad(set_to_none=True)
    npo_history=[]
    for step in range(1,NPO_STEPS+1):
        for _ in range(GRAD_ACCUM):
            forget_index=rng_forget.randrange(len(forget_rows))
            bf=collate([forget_rows[forget_index]])
            br=collate([retain_rows[rng_retain.randrange(len(retain_rows))]])
            ref_lp=torch.tensor([reference_logps[forget_index]], device='cuda')
            pi_lp,_=answer_logps(policy,bf)
            # L_NPO = -(2/beta)*log sigmoid(-beta*(logPpolicy-logPreference))
            npo_loss=-(2.0/BETA)*F.logsigmoid(-BETA*(pi_lp-ref_lp)).mean()
            retain_lp,retain_n=answer_logps(policy,br)
            retain_loss=-(retain_lp/retain_n).mean()
            loss=(npo_loss+RETAIN_COEFF*retain_loss)/GRAD_ACCUM
            if not torch.isfinite(loss): raise RuntimeError('Nonfinite NPO loss')
            loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in policy.parameters() if p.requires_grad], 1.0)
        opt.step();opt.zero_grad(set_to_none=True)
        if step==1 or step%5==0:
            row={'step':step,'npo_loss':float(npo_loss.detach()),'retain_loss':float(retain_loss.detach())}
            npo_history.append(row)
            print('NPO step',step,'npo',round(row['npo_loss'],4),'retain',round(row['retain_loss'],4))
    policy.save_pretrained(str(NPO_ADAPTER))
    tokenizer.save_pretrained(str(NPO_ADAPTER))
    (EXPERIMENT_DIR/'npo_history.json').write_text(json.dumps(npo_history,indent=2))
    del opt,policy,loss,pi_lp,retain_lp,npo_loss,retain_loss;gc.collect();torch.cuda.empty_cache()

    if not npo_history or npo_history[-1]['step'] != NPO_STEPS:
        npo_history.append({'step': NPO_STEPS, 'completed': True})
        (EXPERIMENT_DIR/'npo_history.json').write_text(json.dumps(npo_history, indent=2))


## 7 · Merge the unlearned adapter and save the FP16 checkpoint

In [ ]:
if saved_stage_exists(MERGED_DIR):
    print('Reusing merged FP16 model:', MERGED_DIR)
else:
    if (EXPERIMENT_DIR/'quantized_nf4').exists():
        raise RuntimeError("Quantized model exists without its merge source. Use RUN_MODE='new'.")
    # Merge the NPO adapter into the saved SFT base, preserving SFT in its own folder.
    base_cpu=AutoModelForCausalLM.from_pretrained(str(SFT_MODEL_DIR),torch_dtype=torch.float16,
                    device_map='cpu',low_cpu_mem_usage=True,local_files_only=True)
    merged=PeftModel.from_pretrained(base_cpu,str(NPO_ADAPTER),is_trainable=False).merge_and_unload(safe_merge=True)
    merged.save_pretrained(str(MERGED_DIR),safe_serialization=True,max_shard_size='2GB')
    tokenizer.save_pretrained(str(MERGED_DIR))
    del base_cpu,merged;gc.collect()
    print('Merged FP16 weights saved to:',MERGED_DIR)


## 8 · Ask the same questions after NPO and measure forgetting

The next stage quantizes this same saved checkpoint. Identical evaluation samples and decoding settings are used throughout.

In [10]:
unlearned=AutoModelForCausalLM.from_pretrained(str(MERGED_DIR), torch_dtype=torch.float16,
    device_map='cpu', local_files_only=True, attn_implementation='sdpa').to('cuda')
eval_npo=evaluate(unlearned,'npo_fp16')
del unlearned; gc.collect(); torch.cuda.empty_cache()


Loading weights: 100%|██████████| 146/146 [00:03<00:00, 37.65it/s]


Evaluating npo_fp16 on the fixed question set
npo_fp16 forget05 10/200
npo_fp16 forget05 20/200
npo_fp16 forget05 30/200
npo_fp16 forget05 40/200
npo_fp16 forget05 50/200
npo_fp16 forget05 60/200
npo_fp16 forget05 70/200
npo_fp16 forget05 80/200
npo_fp16 forget05 90/200
npo_fp16 forget05 100/200
npo_fp16 forget05 110/200
npo_fp16 forget05 120/200
npo_fp16 forget05 130/200
npo_fp16 forget05 140/200
npo_fp16 forget05 150/200
npo_fp16 forget05 160/200
npo_fp16 forget05 170/200
npo_fp16 forget05 180/200
npo_fp16 forget05 190/200
npo_fp16 forget05 200/200
npo_fp16 retain95 10/40
npo_fp16 retain95 20/40
npo_fp16 retain95 30/40
npo_fp16 retain95 40/40
npo_fp16 world_facts 10/40
npo_fp16 world_facts 20/40
npo_fp16 world_facts 30/40
npo_fp16 world_facts 40/40

 npo_fp16 
              exact_match  token_f1  answer_nll_per_token
split                                                   
forget05             0.0    0.4305                1.7948
retain95             0.0    0.4414                0.991

## 9 · Quantize the unlearned model, reload it, and ask the same questions again

In [ ]:
QUANTIZED_DIR=EXPERIMENT_DIR/'quantized_nf4'
if not saved_stage_exists(QUANTIZED_DIR):
    quantized=AutoModelForCausalLM.from_pretrained(str(MERGED_DIR),quantization_config=qcfg,
        device_map={'':0},torch_dtype=torch.float16,attn_implementation='sdpa',local_files_only=True)
    quantized.save_pretrained(str(QUANTIZED_DIR))
    tokenizer.save_pretrained(str(QUANTIZED_DIR))
    (QUANTIZED_DIR/'quantization_source.json').write_text(json.dumps({
        'source_checkpoint':str(MERGED_DIR.resolve()),'quantization':qcfg.to_dict()},indent=2),encoding='utf-8')
    del quantized; gc.collect(); torch.cuda.empty_cache()

quantization_source = json.loads((QUANTIZED_DIR/'quantization_source.json').read_text(encoding='utf-8'))
assert Path(quantization_source['source_checkpoint']).resolve() == MERGED_DIR.resolve(), 'NF4 model belongs to a different merged model'
print('Loading saved NF4 model:', QUANTIZED_DIR)
# Test the saved quantized model, with exactly the same prompts and decoding.
quantized=AutoModelForCausalLM.from_pretrained(str(QUANTIZED_DIR),device_map={'':0},
    torch_dtype=torch.float16,attn_implementation='sdpa',local_files_only=True)
assert getattr(quantized,'is_loaded_in_4bit',False), 'Expected a saved 4-bit model'
eval_quant=evaluate(quantized,'npo_merged_nf4')
del quantized; gc.collect(); torch.cuda.empty_cache()


## 10 · Interpret forgetting, quantization recovery, and retained utility

In [12]:
# Paired recovery report version 1. All functions are embedded for standalone use.
"""Paired, interpretable TOFU results; also embedded into the standalone notebook."""
import csv
import hashlib
import html
import json
import math
from collections import Counter, defaultdict
from pathlib import Path
from statistics import mean

STAGES = ('sft', 'npo_fp16', 'npo_merged_nf4')
STAGE_NAMES = ('Fine-tuned', 'After NPO', 'After NF4')


def recovery_transitions(scores, threshold):
    """Conditional recovery: baseline passes -> NPO fails -> NF4 passes."""
    known = [row for row in scores if row[0] >= threshold]
    lost = [row for row in known if row[1] < threshold]
    returned = [row for row in lost if row[2] >= threshold]
    return dict(baseline_pass=len(known), lost_after_npo=len(lost), returned_after_nf4=len(returned),
                forgetting_rate=len(lost)/len(known) if known else None,
                recovery_rate=len(returned)/len(lost) if lost else None)


def paired_rows(records):
    indexed = {stage: {} for stage in STAGES}
    for record in records:
        stage = record['model']
        if stage not in indexed:
            raise ValueError(f'Unexpected model stage: {stage}')
        key = (record['split'], record['question'])
        if key in indexed[stage]:
            raise ValueError(f'Duplicate evaluation question in {stage}: {key}')
        row = dict(record)
        for metric in ('exact_match', 'token_f1', 'answer_nll_per_token'):
            row[metric] = float(row[metric])
            if not math.isfinite(row[metric]):
                raise ValueError(f'Nonfinite {metric}')
        if row['exact_match'] not in (0, 1) or not 0 <= row['token_f1'] <= 1 or row['answer_nll_per_token'] < 0:
            raise ValueError('Invalid evaluation metric range')
        indexed[stage][key] = row
    keys = set(indexed['sft'])
    if not keys or any(set(indexed[s]) != keys for s in STAGES):
        raise ValueError('Stages must contain exactly the same questions and splits; no silent inner joins')
    result = []
    for key in sorted(keys):
        triplet = [indexed[s][key] for s in STAGES]
        if len({r['reference'] for r in triplet}) != 1:
            raise ValueError(f'Reference answer changed across stages: {key}')
        q = dict(split=key[0], question=key[1], reference=triplet[0]['reference'])
        q['question_id'] = hashlib.sha256(json.dumps([key[0],key[1],q['reference']],ensure_ascii=False).encode()).hexdigest()[:16]
        for stage, row in zip(STAGES, triplet):
            for field in ('prediction', 'exact_match', 'token_f1', 'answer_nll_per_token'):
                q[f'{field}_{stage}'] = row[field]
        exact = [q[f'exact_match_{s}'] for s in STAGES]
        if not exact[0]:
            q['exact_status'] = 'Not exact at baseline'
        elif exact[1]:
            q['exact_status'] = 'Exact answer persisted after NPO'
        elif exact[2]:
            q['exact_status'] = 'Exact answer reappeared after NF4'
        else:
            q['exact_status'] = 'Exact answer lost; no exact recovery'
        q['nll_increase_after_npo'] = q['answer_nll_per_token_npo_fp16'] - q['answer_nll_per_token_sft']
        q['nll_decrease_after_nf4'] = q['answer_nll_per_token_npo_fp16'] - q['answer_nll_per_token_npo_merged_nf4']
        q['f1_drop_after_npo'] = q['token_f1_sft'] - q['token_f1_npo_fp16']
        q['f1_gain_after_nf4'] = q['token_f1_npo_merged_nf4'] - q['token_f1_npo_fp16']
        result.append(q)
    return result


def summarize_recovery(paired):
    groups = defaultdict(list)
    for row in paired:
        groups[row['split']].append(row)
    stage_summary = []
    for split, rows in groups.items():
        for stage in STAGES:
            item = dict(split=split, model=stage, count=len(rows))
            for field in ('exact_match','token_f1','answer_nll_per_token'):
                item[field] = mean(r[f'{field}_{stage}'] for r in rows)
            stage_summary.append(item)
    forget = [r for r in paired if r['split'].startswith('forget')]
    if not forget:
        raise ValueError('No forget split in evaluation')
    if len({r['split'] for r in forget}) != 1:
        raise ValueError('Do not pool different forgetting experiments')
    exact = recovery_transitions([[r[f'exact_match_{s}'] for s in STAGES] for r in forget], 1)
    sensitivity = {str(t): recovery_transitions([[r[f'token_f1_{s}'] for s in STAGES] for r in forget], t)
                   for t in (0.6,0.8,0.9)}
    sft_nll,npo_nll,q_nll = [mean(r[f'answer_nll_per_token_{s}'] for r in forget) for s in STAGES]
    sft_f1,npo_f1,q_f1 = [mean(r[f'token_f1_{s}'] for r in forget) for s in STAGES]
    nll_gap=npo_nll-sft_nll
    f1_gap=sft_f1-npo_f1
    return dict(forget_split=forget[0]['split'],forget_count=len(forget),stages=stage_summary,
        exact=exact,overlap_sensitivity=sensitivity,
        nll_increase_after_npo=nll_gap,nll_decrease_after_nf4=npo_nll-q_nll,
        nll_shift_reversed_fraction=(npo_nll-q_nll)/nll_gap if nll_gap>1e-8 else None,
        f1_drop_after_npo=f1_gap,f1_gain_after_nf4=q_f1-npo_f1,
        f1_drop_reversed_fraction=(q_f1-npo_f1)/f1_gap if f1_gap>1e-8 else None,
        forget_questions_with_nll_increase=sum(r['nll_increase_after_npo']>0 for r in forget),
        forget_questions_with_nll_decrease_after_nf4=sum(r['nll_decrease_after_nf4']>0 for r in forget))


def _percent(value):
    return 'n/a (zero denominator)' if value is None else f'{100*value:.2f}%'


def report_text(report, run_dir):
    exact=report['exact']
    lines=['# TOFU forgetting and quantization recovery', '',
        f'Run: `{Path(run_dir).name}`. Forget split: **{report["forget_split"]}**, **{report["forget_count"]} evaluated questions**.', '',
        '**Workflow:** local model → fine-tune on full TOFU → save/reload SFT → evaluate → NPO → evaluate → NF4 quantization → ask the same questions again.', '',
        '## Did it remember, forget, and then remember again?', '',
        f'- Before NPO, **{exact["baseline_pass"]}/{report["forget_count"]}** answers matched the reference exactly.',
        f'- NPO removed exact matches on **{exact["lost_after_npo"]}/{exact["baseline_pass"]}** of those previously exact answers ({_percent(exact["forgetting_rate"])}).',
        f'- After NF4, **{exact["returned_after_nf4"]}/{exact["lost_after_npo"]}** of those lost exact answers reappeared ({_percent(exact["recovery_rate"])}).', '',
        '**These are exact-answer transitions, not a count of all facts stored in the model.** Correct paraphrases can fail exact matching. Questions already answered incorrectly before NPO cannot establish successful forgetting.', '',
        '## Score changes on the forget set', '',
        f'- NPO increased target-answer NLL by **{report["nll_increase_after_npo"]:.4f}**. Higher NLL means the reference answer became less likely.',
        f'- NF4 then decreased NLL by **{report["nll_decrease_after_nf4"]:.4f}**: **{_percent(report["nll_shift_reversed_fraction"])} of the NLL shift was reversed**.',
        f'- Token F1 fell by **{100*report["f1_drop_after_npo"]:.2f} percentage points** after NPO and changed by **{100*report["f1_gain_after_nf4"]:+.2f} points** after NF4.',
        f'- **{_percent(report["f1_drop_reversed_fraction"])} of the mean F1 drop was reversed** after NF4.', '',
        'The reversal percentages describe changes in scores; they are **not percentages of forgotten facts recovered**. They are signed and are not clipped: negative means further degradation, above 100% means the score exceeded its SFT level. They are n/a when there was no preceding degradation to reverse.', '',
        '## Forgetting versus retained utility', '',
        '| Set | Stage | Questions | Exact match | Token F1 | Answer NLL/token |',
        '|---|---|---:|---:|---:|---:|']
    for row in report['stages']:
        lines.append(f'| {row["split"]} | {STAGE_NAMES[STAGES.index(row["model"])]} | {row["count"]} | {100*row["exact_match"]:.2f}% | {row["token_f1"]:.4f} | {row["answer_nll_per_token"]:.4f} |')
    lines += ['', 'Lower forget recall alongside stable retain scores is the intended pattern. If retain F1 also falls or its NLL increases, unlearning has damaged retained utility.', '',
        '## Word-overlap sensitivity — heuristic only', '',
        'A high F1 score can still contain wrong names or dates. These thresholds flag examples for inspection, not semantic correctness.', '',
        '| F1 threshold | High overlap at SFT | Fell below after NPO | Crossed back after NF4 | Conditional return rate |',
        '|---:|---:|---:|---:|---:|']
    for threshold, row in report['overlap_sensitivity'].items():
        lines.append(f'| {threshold} | {row["baseline_pass"]} | {row["lost_after_npo"]} | {row["returned_after_nf4"]} | {_percent(row["recovery_rate"])} |')
    lines += ['', '## Scope and limits', '',
        f'- Forget-set coverage: {report.get("coverage_note", "dataset snapshot unavailable")}.',
        '- Open the HTML report to inspect every question, its reference, and all three generated answers. Use that comparison to check factual correctness.',
        '- The same questions and greedy decoding are paired across stages. Training-set recall is intentional here; this is not a held-out generalization score.',
        '- This is one run, one NF4 quantizer, and one direct prompt per question. Paraphrases, stronger extraction attacks, and other quantizers may behave differently.',
        '- This is not official TOFU forget quality. That requires additional benchmark metrics and a retain-only comparator. No proof of complete information deletion follows.',
        '- Word-overlap scores may reward copied question wording. World-facts results here use direct-answer generation, not the official multiple-choice normalization.', '',
        'Method reference: [TOFU evaluation](https://locuslab.github.io/tofu/).']
    if report.get('evaluation_settings'):
        settings=report['evaluation_settings']
        lines.insert(-2, f'- Saved evaluation settings: maximum input length {settings.get("max_length","unrecorded")} tokens; maximum generated answer length {settings.get("eval_new_tokens","unrecorded")} tokens. Length limits can affect matching and likelihood scores.')
    if report.get('smoke_only'):
        lines.insert(2, '**SMOKE TEST ONLY: insufficient training/evaluation for a forgetting claim.**')
    return '\n'.join(lines)+'\n'


def report_html(report, paired, markdown):
    esc=html.escape
    def table(headers, rows):
        return '<div class="table-scroll"><table><thead><tr>'+''.join(f'<th>{esc(str(x))}</th>' for x in headers)+'</tr></thead><tbody>'+''.join('<tr>'+''.join(f'<td>{esc(str(x))}</td>' for x in row)+'</tr>' for row in rows)+'</tbody></table></div>'
    exact=report['exact']
    metrics=[('Exact at SFT',f'{exact["baseline_pass"]}/{report["forget_count"]}'),
        ('Lost after NPO',f'{exact["lost_after_npo"]}/{exact["baseline_pass"]}'),
        ('Lost exact answers returning after NF4',f'{exact["returned_after_nf4"]}/{exact["lost_after_npo"]}'),
        ('NLL shift reversed',_percent(report['nll_shift_reversed_fraction']))]
    summary_table=table(['Set','Stage','N','Exact match','Token F1','NLL/token'],[
        [r['split'],STAGE_NAMES[STAGES.index(r['model'])],r['count'],f'{100*r["exact_match"]:.2f}%',f'{r["token_f1"]:.4f}',f'{r["answer_nll_per_token"]:.4f}'] for r in report['stages']])
    sensitivity=table(['F1 threshold','High at SFT','Dropped after NPO','Returned after NF4','Return rate'],[
        [t,r['baseline_pass'],r['lost_after_npo'],r['returned_after_nf4'],_percent(r['recovery_rate'])] for t,r in report['overlap_sensitivity'].items()])
    cards=[]
    for row in paired:
        stages=''.join(f'<section><h4>{name}</h4><p class="answer">{esc(row[f"prediction_{stage}"])}</p><p class="score">Exact: {int(row[f"exact_match_{stage}"])} · F1: {row[f"token_f1_{stage}"]:.3f} · NLL: {row[f"answer_nll_per_token_{stage}"]:.3f}</p></section>' for stage,name in zip(STAGES,STAGE_NAMES))
        cards.append(f'<article class="question" data-split="{esc(row["split"],quote=True)}" data-status="{esc(row["exact_status"],quote=True)}"><p class="tag">{esc(row["split"])} · {esc(row["exact_status"])}</p><h3>{esc(row["question"])}</h3><p class="reference"><strong>Reference:</strong> {esc(row["reference"])}</p><div class="answers">{stages}</div></article>')
    splits=''.join(f'<option>{esc(s)}</option>' for s in sorted({r['split'] for r in paired}))
    statuses=''.join(f'<option>{esc(s)}</option>' for s in sorted({r['exact_status'] for r in paired}))
    return '''<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>TOFU forgetting and recovery</title>
<style>body{margin:0;background:#f4f6fa;color:#172536;font:16px/1.55 system-ui,sans-serif}main{max-width:1240px;margin:auto;padding:32px}h1{font-size:32px;line-height:1.2}h2{margin-top:36px}h3{font-size:19px}h4{margin:0;color:#244b71}.muted,.score{color:#596777;font-size:14px}.flow{padding:16px;background:#e9eff8;border-radius:8px}.metrics{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin:24px 0}.metric,.question{background:white;border:1px solid #dbe2eb;border-radius:10px;padding:20px}.metric strong{display:block;font-size:27px}.metric span{font-size:14px}.notice{padding:16px;border-left:4px solid #c78b25;background:#fff7e6}.table-scroll{overflow:auto}table{border-collapse:collapse;background:white;width:100%;font-size:14px}th,td{padding:11px 14px;border-bottom:1px solid #dde3eb;text-align:left}th{background:#e9eff8}td:nth-child(n+3){font-variant-numeric:tabular-nums}.filters{display:flex;flex-wrap:wrap;gap:10px;margin:20px 0}input,select{font:inherit;padding:10px;border:1px solid #bac6d5;border-radius:6px}input{min-width:260px;flex:1}.question{margin:16px 0}.answers{display:grid;grid-template-columns:repeat(3,1fr);gap:18px}.answers section{padding:14px;background:#f7f9fc;border-radius:6px}.answer{white-space:pre-wrap;overflow-wrap:anywhere}.reference{background:#edf7f2;padding:12px}.tag{font-size:13px;color:#53667b}.score{font-variant-numeric:tabular-nums}pre{white-space:pre-wrap;overflow-wrap:anywhere;background:white;padding:20px;font:14px/1.5 system-ui}a{color:#195fa3}[hidden]{display:none!important}@media(max-width:800px){main{padding:16px}.metrics{grid-template-columns:1fr 1fr}.answers{grid-template-columns:1fr}}</style></head><body><main>
<h1>Did forgetting survive quantization?</h1><p class="flow">Load local model → fine-tune on full TOFU → evaluate → NPO → evaluate forgetting → NF4 → ask the same questions again</p>'''+f'<p class="muted">{esc(report["forget_split"])} · {report["forget_count"]} forget questions · {esc(report.get("coverage_note",""))}</p>'+('<p class="notice">Smoke test only — do not interpret as evidence of forgetting.</p>' if report.get('smoke_only') else '')+'<div class="metrics">'+''.join(f'<div class="metric"><strong>{esc(value)}</strong><span>{esc(label)}</span></div>' for label,value in metrics)+'</div>'+'''
<p class="notice"><strong>Read the labels carefully.</strong> Exact matching is strict and misses correct paraphrases. Token F1 is word overlap and can reward factually wrong answers. NLL measures target-answer likelihood. A percentage reversal of a score is not a percentage of memories recovered. Recovery counts only questions that passed at SFT and failed after NPO.</p>
<h2>Compare all three checkpoints</h2><p>Forgetting should reduce recall on the forget set while preserving retained utility. On the forget set, higher NLL and lower F1 support reduced recall. After NF4, the opposite change is a recovery signal.</p>'''+summary_table+'<h2>Overlap threshold sensitivity</h2><p>Exploratory flags for manual inspection. These are not factual-correctness verdicts.</p>'+sensitivity+'''
<h2>Inspect the actual answers</h2><p>Compare names, dates, book titles, and other factual details against the reference. The default view shows every evaluated forget question.</p><div class="filters"><input id="search" aria-label="Search questions and answers" placeholder="Search a question, author, or answer"><select id="split" aria-label="Dataset split"><option value="">All sets</option>'''+splits+'</select><select id="status" aria-label="Exact-answer transition"><option value="">All exact-answer transitions</option>'+statuses+'</select></div><p id="count" class="muted"></p>'+''.join(cards)+'<h2>Definitions and limitations</h2><pre>'+esc(markdown)+'</pre><p><a href="question_comparison.csv">Download per-question comparison</a> · <a href="summary.json">Summary data</a> · <a href="RESULTS.md">Readable text report</a></p></main>'+'''
<script>const cards=[...document.querySelectorAll('.question')];const search=document.querySelector('#search'),split=document.querySelector('#split'),status=document.querySelector('#status');split.value='''+json.dumps(report['forget_split']).replace('<','\u003c')+''';function filter(){let n=0;for(const card of cards){const show=(!split.value||card.dataset.split===split.value)&&(!status.value||card.dataset.status===status.value)&&card.textContent.toLowerCase().includes(search.value.toLowerCase());card.hidden=!show;if(show)n++;}document.querySelector('#count').textContent=n+' questions shown';}for(const control of [search,split,status])control.addEventListener('input',filter);filter();</script></body></html>'''


def make_report(run_dir, output_dir=None):
    run_dir=Path(run_dir)
    output_dir=Path(output_dir) if output_dir else run_dir/'interpretation'
    records=[]
    for stage in STAGES:
        with (run_dir/f'eval_{stage}.csv').open(encoding='utf-8',newline='') as handle:
            rows=list(csv.DictReader(handle))
        if any(r['model']!=stage for r in rows):
            raise ValueError(f'Mislabelled evaluation file for {stage}')
        records.extend(rows)
    paired=paired_rows(records)
    report=summarize_recovery(paired)
    settings_path=run_dir/'settings.json'
    saved_settings=json.loads(settings_path.read_text(encoding='utf-8')) if settings_path.exists() else {}
    report['smoke_only']=saved_settings.get('smoke_only')
    report['evaluation_settings']={k:saved_settings[k] for k in ('max_length','eval_new_tokens','seed') if k in saved_settings}
    snapshot=run_dir/f'dataset_{report["forget_split"]}.jsonl'
    if snapshot.exists():
        snapshot_rows=[json.loads(line) for line in snapshot.read_text(encoding='utf-8').splitlines() if line.strip()]
        source_keys={(r['question'],r['answer']) for r in snapshot_rows}
        evaluated_keys={(r['question'],r['reference']) for r in paired if r['split']==report['forget_split']}
        if not evaluated_keys <= source_keys:
            raise ValueError('Forget evaluations do not match this run\'s dataset snapshot')
        report['coverage_note']=f'{len(evaluated_keys)}/{len(source_keys)} unique forget questions evaluated'+(' (complete forget set)' if evaluated_keys==source_keys else ' (subset only)')
    else:
        report['coverage_note']='source snapshot unavailable; full-set coverage unverified'
    report['sources']=[dict(file=f'eval_{s}.csv',sha256=hashlib.sha256((run_dir/f'eval_{s}.csv').read_bytes()).hexdigest()) for s in STAGES]
    output_dir.mkdir(parents=True,exist_ok=True)
    (output_dir/'summary.json').write_text(json.dumps(report,indent=2,allow_nan=False),encoding='utf-8')
    with (output_dir/'question_comparison.csv').open('w',encoding='utf-8',newline='') as handle:
        writer=csv.DictWriter(handle,fieldnames=list(paired[0]))
        writer.writeheader();writer.writerows(paired)
    with (output_dir/'stage_comparison.csv').open('w',encoding='utf-8',newline='') as handle:
        writer=csv.DictWriter(handle,fieldnames=list(report['stages'][0]))
        writer.writeheader();writer.writerows(report['stages'])
    markdown=report_text(report,run_dir)
    (output_dir/'RESULTS.md').write_text(markdown,encoding='utf-8')
    (output_dir/'report.html').write_text(report_html(report,paired,markdown),encoding='utf-8')
    return report



report=make_report(EXPERIMENT_DIR)
from IPython.display import Markdown,display
display(Markdown((EXPERIMENT_DIR/'interpretation'/'RESULTS.md').read_text(encoding='utf-8')))
print('Open the searchable side-by-side answer report:',EXPERIMENT_DIR/'interpretation'/'report.html')


# TOFU forgetting and quantization recovery

Run: `local_tofu_npo_20261025_150047_128001`. Forget split: **forget05**, **200 evaluated questions**.

**Workflow:** local model → fine-tune on full TOFU → save/reload SFT → evaluate → NPO → evaluate → NF4 quantization → ask the same questions again.

## Did it remember, forget, and then remember again?

- Before NPO, **197/200** answers matched the reference exactly.
- NPO removed exact matches on **197/197** of those previously exact answers (100.00%).
- After NF4, **0/197** of those lost exact answers reappeared (0.00%).

**These are exact-answer transitions, not a count of all facts stored in the model.** Correct paraphrases can fail exact matching. Questions already answered incorrectly before NPO cannot establish successful forgetting.

## Score changes on the forget set

- NPO increased target-answer NLL by **1.7805**. Higher NLL means the reference answer became less likely.
- NF4 then decreased NLL by **-0.5015**: **-28.17% of the NLL shift was reversed**.
- Token F1 fell by **56.80 percentage points** after NPO and changed by **-1.75 points** after NF4.
- **-3.09% of the mean F1 drop was reversed** after NF4.

The reversal percentages describe changes in scores; they are **not percentages of forgotten facts recovered**. They are signed and are not clipped: negative means further degradation, above 100% means the score exceeded its SFT level. They are n/a when there was no preceding degradation to reverse.

## Forgetting versus retained utility

| Set | Stage | Questions | Exact match | Token F1 | Answer NLL/token |
|---|---|---:|---:|---:|---:|
| forget05 | Fine-tuned | 200 | 98.50% | 0.9984 | 0.0143 |
| forget05 | After NPO | 200 | 0.00% | 0.4305 | 1.7948 |
| forget05 | After NF4 | 200 | 0.00% | 0.4129 | 2.2963 |
| retain95 | Fine-tuned | 40 | 97.50% | 0.9995 | 0.0157 |
| retain95 | After NPO | 40 | 0.00% | 0.4414 | 0.9910 |
| retain95 | After NF4 | 40 | 0.00% | 0.3998 | 1.8547 |
| world_facts | Fine-tuned | 40 | 0.00% | 0.1597 | 13.0099 |
| world_facts | After NPO | 40 | 0.00% | 0.1233 | 17.1318 |
| world_facts | After NF4 | 40 | 0.00% | 0.1363 | 16.1255 |

Lower forget recall alongside stable retain scores is the intended pattern. If retain F1 also falls or its NLL increases, unlearning has damaged retained utility.

## Word-overlap sensitivity — heuristic only

A high F1 score can still contain wrong names or dates. These thresholds flag examples for inspection, not semantic correctness.

| F1 threshold | High overlap at SFT | Fell below after NPO | Crossed back after NF4 | Conditional return rate |
|---:|---:|---:|---:|---:|
| 0.6 | 200 | 166 | 12 | 7.23% |
| 0.8 | 200 | 194 | 4 | 2.06% |
| 0.9 | 198 | 196 | 2 | 1.02% |

## Scope and limits

- Forget-set coverage: 200/200 unique forget questions evaluated (complete forget set).
- Open the HTML report to inspect every question, its reference, and all three generated answers. Use that comparison to check factual correctness.
- The same questions and greedy decoding are paired across stages. Training-set recall is intentional here; this is not a held-out generalization score.
- This is one run, one NF4 quantizer, and one direct prompt per question. Paraphrases, stronger extraction attacks, and other quantizers may behave differently.
- This is not official TOFU forget quality. That requires additional benchmark metrics and a retain-only comparator. No proof of complete information deletion follows.
- Word-overlap scores may reward copied question wording. World-facts results here use direct-answer generation, not the official multiple-choice normalization.
- Saved evaluation settings: maximum input length 256 tokens; maximum generated answer length 128 tokens. Length limits can affect matching and likelihood scores.

Method reference: [TOFU evaluation](https://locuslab.github.io/tofu/).


Open the searchable side-by-side answer report: c:\MY-FYP\fq-ssiuu\runs\local_tofu_npo_20261025_150047_128001\interpretation\report.html


## How to read the result

- **Exact-answer recovery:** among answers that were exact before NPO and stopped being exact after NPO, how many become exact again after NF4? The report prints the numerator and denominator.
- **Likelihood/F1 reversal:** how much of the score change introduced by NPO did quantization reverse? This is a change in a score, not the percentage of facts recovered.
- **Factual inspection:** open the HTML report and compare each reference with all three generated answers. Word overlap can be high even when a name, date, or book title is wrong.
- **Retained utility:** check the retain/world-facts scores too. A model that loses knowledge everywhere has not selectively forgotten.
- All forget questions are evaluated by default. Exact matching can miss correct paraphrases; direct prompting does not prove deletion or resistance to stronger extraction attacks.

These are local experiment metrics, not official TOFU forget quality. See [TOFU's evaluation definitions](https://locuslab.github.io/tofu/).
